In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Lambda, Multiply, concatenate, Dot
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# File yang dibutuhkan di Kaggle dataset:
#   ae_answers_emb.npy   <- output balancing_data_klasifikasi.ipynb
#   ae_metadata.pkl      <- output balancing_data_klasifikasi.ipynb
#   questions_emb.npy    <- output fasttext.ipynb (per IDPSJ, tidak berubah)
#   answerkeys_emb.npy   <- output fasttext.ipynb (per IDPSJ, tidak berubah)

DATASET_SLUG = "siamese-data"
DATA_DIR     = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR      = "/kaggle/working"

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

for fname in ['ae_answers_emb.npy', 'ae_metadata.pkl',
              'questions_emb.npy', 'answerkeys_emb.npy']:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<32} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# ae_answers_emb.npy  : answers setelah Mixup A-E (label_ae sudah ada di metadata)
# ae_metadata.pkl     : metadata dengan kolom label_ae, psj_idx, is_synthetic
# questions/answerkeys_emb.npy : per IDPSJ, langsung dari fasttext.ipynb

answers_emb    = np.load(os.path.join(DATA_DIR, 'ae_answers_emb.npy'))
uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'ae_metadata.pkl'))
metadata       = metadata.reset_index(drop=True)

# Rekonstruksi questions & answerkeys per sampel menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

# Filter data asli (non-sintetis) — digunakan untuk val & test
is_real = ~metadata['is_synthetic'].values

# Konstanta label A-E — dipakai di semua cell berikutnya
LABEL_TO_IDX = {'E': 0, 'D': 1, 'C': 2, 'B': 3, 'A': 4}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}
N_CLASSES    = 5

print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows  (asli={is_real.sum()}, sintetis={(~is_real).sum()})")
print(f"Kolom          : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")

print("\nDistribusi label A-E (semua data):")
cnt = metadata['label_ae'].value_counts().sort_index()
for idx, n in cnt.items():
    print(f"  {IDX_TO_LABEL[idx]} (idx={idx}): {n}")
print(f"Ratio max/min: {cnt.max() / cnt.min():.2f}x")

print("\nDistribusi label A-E (data asli saja):")
cnt_r = metadata.loc[is_real, 'label_ae'].value_counts().sort_index()
for idx, n in cnt_r.items():
    print(f"  {IDX_TO_LABEL[idx]} (idx={idx}): {n}")

In [ ]:
# ── Hyperparameter & Build Model ──────────────────────────────────────────────
BILSTM_UNITS = 128
DROPOUT      = 0.3
EPOCHS       = 150
BATCH_SIZE   = 32
PATIENCE     = 15
LR           = 1e-3
N_SEEDS      = 3
DENSE1       = 256
DENSE2       = 64


def build_model_clf(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                    bilstm_units=BILSTM_UNITS, dropout=DROPOUT,
                    dense1=DENSE1, dense2=DENSE2, n_classes=N_CLASSES):
    """
    Siamese BiLSTM — Klasifikasi A-E (5 kelas)

    Arsitektur feature extraction sama dengan versi regresi.
    Output: Dense(5, softmax) -> label E=0, D=1, C=2, B=3, A=4
    Loss  : sparse_categorical_crossentropy
    """
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=False), name='bilstm_shared'
    )

    inp_q  = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a  = Input(shape=(a_seq_len,  emb_dim), name='inp_a')

    raw_ak_mean = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_ak_mean')(inp_ak)
    raw_a_mean  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_a_mean')(inp_a)
    raw_q_mean  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_q_mean')(inp_q)

    raw_cos_ak_a = Dot(axes=1, normalize=True, name='raw_cos_ak_a')([raw_ak_mean, raw_a_mean])
    raw_cos_q_a  = Dot(axes=1, normalize=True, name='raw_cos_q_a')([raw_q_mean, raw_a_mean])

    a_len  = Lambda(
        lambda x: tf.reduce_sum(
            tf.cast(tf.reduce_any(tf.abs(x) > 1e-6, axis=-1), tf.float32),
            axis=-1, keepdims=True),
        name='a_len')(inp_a)
    ak_len = Lambda(
        lambda x: tf.reduce_sum(
            tf.cast(tf.reduce_any(tf.abs(x) > 1e-6, axis=-1), tf.float32),
            axis=-1, keepdims=True),
        name='ak_len')(inp_ak)
    raw_len_ratio = Lambda(
        lambda x: x[0] / (x[1] + 1e-8), name='raw_len_ratio'
    )([a_len, ak_len])

    eak = shared_bilstm(inp_ak)
    ea  = shared_bilstm(inp_a)

    abs_diff     = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    merged = concatenate(
        [ea, eak, abs_diff, had_prod,
         raw_cos_ak_a, raw_cos_q_a, cos_sim_ak_a, raw_len_ratio],
        name='merged'
    )

    x   = Dense(dense1, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(dense2, activation='relu')(x)
    out = Dense(n_classes, activation='softmax', name='output')(x)

    model = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out, name='siamese_bilstm_ae')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


_tmp = build_model_clf(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2]
)
_tmp.summary()
del _tmp

In [ ]:
# ── Hyperparameter Search (Optuna) ────────────────────────────────────────────
# Metrik optimasi: accuracy (maximize, direction='maximize').

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEARCH_FOLD_INDICES = [0, 3, 6, 9]
SEARCH_EPOCHS       = 80
SEARCH_PATIENCE     = 8
N_TRIALS            = 40

idpsj_list_search = sorted(metadata['IDPSJ'].unique())
y_ae_all          = metadata['label_ae'].values.astype(np.int32)  # 0-4 (E..A)


def run_one_fold_clf(fold_i, bilstm_units, dropout, lr, batch_size, dense1, dense2):
    test_id   = idpsj_list_search[fold_i]
    val_id    = idpsj_list_search[(fold_i + 1) % len(idpsj_list_search)]
    train_ids = [p for p in idpsj_list_search if p != test_id and p != val_id]

    tr_idx  = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    te_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    def gs(arr, idx): return arr[idx].astype(np.float32)

    y_tr = y_ae_all[tr_idx]
    y_v  = y_ae_all[val_idx]
    y_te = y_ae_all[te_idx]

    ug, uc = np.unique(y_tr, return_counts=True)
    fm     = dict(zip(ug, uc))
    raw_w  = np.array([len(y_tr) / (len(ug) * fm[g]) for g in y_tr])
    sw     = raw_w / raw_w.mean()

    shared = Bidirectional(LSTM(bilstm_units, return_sequences=False), name='bilstm_shared')
    inp_q  = Input(shape=(questions_emb.shape[1],  answers_emb.shape[2]), name='inp_q')
    inp_ak = Input(shape=(answerkeys_emb.shape[1], answers_emb.shape[2]), name='inp_ak')
    inp_a  = Input(shape=(answers_emb.shape[1],    answers_emb.shape[2]), name='inp_a')

    raw_ak_m = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_ak_mean')(inp_ak)
    raw_a_m  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_a_mean')(inp_a)
    raw_q_m  = Lambda(lambda x: tf.reduce_mean(x, axis=1), name='raw_q_mean')(inp_q)
    rcos_ak  = Dot(axes=1, normalize=True, name='raw_cos_ak_a')([raw_ak_m, raw_a_m])
    rcos_q   = Dot(axes=1, normalize=True, name='raw_cos_q_a')([raw_q_m, raw_a_m])

    a_l  = Lambda(lambda x: tf.reduce_sum(
                    tf.cast(tf.reduce_any(tf.abs(x) > 1e-6, axis=-1), tf.float32),
                    axis=-1, keepdims=True), name='a_len')(inp_a)
    ak_l = Lambda(lambda x: tf.reduce_sum(
                    tf.cast(tf.reduce_any(tf.abs(x) > 1e-6, axis=-1), tf.float32),
                    axis=-1, keepdims=True), name='ak_len')(inp_ak)
    rlr  = Lambda(lambda x: x[0] / (x[1] + 1e-8), name='raw_len_ratio')([a_l, ak_l])

    eak = shared(inp_ak)
    ea  = shared(inp_a)
    abd = Lambda(lambda x: tf.abs(x[0] - x[1]), name='abs_diff')([eak, ea])
    had = Multiply(name='had_prod')([eak, ea])
    cos = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])

    mg  = concatenate([ea, eak, abd, had, rcos_ak, rcos_q, cos, rlr], name='merged')
    x   = Dense(dense1, activation='relu')(mg)
    x   = Dropout(dropout)(x)
    x   = Dense(dense2, activation='relu')(x)
    out = Dense(N_CLASSES, activation='softmax')(x)

    m = Model(inputs=[inp_q, inp_ak, inp_a], outputs=out)
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    cb = [
        EarlyStopping(monitor='val_accuracy', patience=SEARCH_PATIENCE,
                      restore_best_weights=True, verbose=0, mode='max'),
        ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3,
                          min_lr=1e-6, verbose=0, mode='max')
    ]
    m.fit(
        [gs(questions_emb, tr_idx), gs(answerkeys_emb, tr_idx), gs(answers_emb, tr_idx)], y_tr,
        sample_weight=sw,
        validation_data=(
            [gs(questions_emb, val_idx), gs(answerkeys_emb, val_idx), gs(answers_emb, val_idx)], y_v
        ),
        epochs=SEARCH_EPOCHS, batch_size=batch_size, callbacks=cb, verbose=0
    )

    yp  = np.argmax(
        m.predict([gs(questions_emb, te_idx), gs(answerkeys_emb, te_idx),
                   gs(answers_emb, te_idx)], verbose=0),
        axis=1
    )
    acc = accuracy_score(y_te, yp)
    tf.keras.backend.clear_session()
    return acc


def objective(trial):
    bilstm_units = trial.suggest_categorical('bilstm_units', [64, 128, 192, 256])
    dropout      = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    lr           = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [16, 32, 64])
    dense1       = trial.suggest_categorical('dense1', [128, 256, 512])
    dense2       = trial.suggest_categorical('dense2', [32, 64, 128])

    accs = []
    for fi in SEARCH_FOLD_INDICES:
        try:
            acc = run_one_fold_clf(fi, bilstm_units, dropout, lr, batch_size, dense1, dense2)
            accs.append(acc)
        except Exception as e:
            print(f"  Trial gagal fold {fi}: {e}")
            return 0.0
    mean_acc = np.mean(accs)
    print(f"  Trial {trial.number:3d} | bilstm={bilstm_units} drop={dropout:.2f} "
          f"lr={lr:.1e} bs={batch_size} d1={dense1} d2={dense2} -> Acc={mean_acc:.4f}")
    return mean_acc


print(f"Memulai Optuna search: {N_TRIALS} trial x {len(SEARCH_FOLD_INDICES)} fold")
study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

best = study.best_params
print(f"\n{'='*55}")
print(f"Hyperparameter Terbaik (Acc={study.best_value:.4f}):")
print(f"  BILSTM_UNITS = {best['bilstm_units']}")
print(f"  DROPOUT      = {best['dropout']}")
print(f"  LR           = {best['lr']:.2e}")
print(f"  BATCH_SIZE   = {best['batch_size']}")
print(f"  Dense(1)     = {best['dense1']}")
print(f"  Dense(2)     = {best['dense2']}")
print(f"{'='*55}")
print("Perbarui konstanta di cell hyperparameter dengan nilai di atas.")

top5 = (
    study.trials_dataframe()
    .sort_values('value', ascending=False)
    .head(5)
    [['number', 'value', 'params_bilstm_units', 'params_dropout',
      'params_lr', 'params_batch_size', 'params_dense1', 'params_dense2']]
)
print("\nTop-5 trial:")
print(top5.to_string(index=False))

In [ ]:
# ── LOPO Cross-Validation + Ensemble (Klasifikasi A-E) ────────────────────────
# Label: E=0, D=1, C=2, B=3, A=4  (dari kolom label_ae di metadata)
# Ensemble: rata-rata probabilitas (n_test, 5) antar seed, lalu argmax.

idpsj_list = sorted(metadata['IDPSJ'].unique())
n_parts    = len(idpsj_list)
y_ae_all   = metadata['label_ae'].values.astype(np.int32)  # 0-4

fold_results = []

for i, test_id in enumerate(idpsj_list):
    val_id    = idpsj_list[(i + 1) % n_parts]
    train_ids = [pid for pid in idpsj_list if pid != test_id and pid != val_id]

    print(f"\n{'='*60}")
    print(f"Fold {i+1:02d}/{n_parts}  |  Test={test_id}  |  Val={val_id}")

    train_idx = metadata.index[metadata['IDPSJ'].isin(train_ids)].values
    val_idx   = metadata.index[(metadata['IDPSJ'] == val_id)  & is_real].values
    test_idx  = metadata.index[(metadata['IDPSJ'] == test_id) & is_real].values

    y_train = y_ae_all[train_idx]
    y_val   = y_ae_all[val_idx]
    y_test  = y_ae_all[test_idx]

    print(f"  Train: {len(y_train)}  |  Val: {len(y_val)}  |  Test: {len(y_test)}")
    train_dist = {IDX_TO_LABEL[k]: v for k, v in
                  zip(*np.unique(y_train, return_counts=True))}
    print(f"  Distribusi train: {train_dist}")

    def get_split(arr, idx):
        return arr[idx].astype(np.float32)

    X_q_tr   = get_split(questions_emb,  train_idx)
    X_ak_tr  = get_split(answerkeys_emb, train_idx)
    X_a_tr   = get_split(answers_emb,    train_idx)
    X_q_val  = get_split(questions_emb,  val_idx)
    X_ak_val = get_split(answerkeys_emb, val_idx)
    X_a_val  = get_split(answers_emb,    val_idx)
    X_q_te   = get_split(questions_emb,  test_idx)
    X_ak_te  = get_split(answerkeys_emb, test_idx)
    X_a_te   = get_split(answers_emb,    test_idx)

    # Sample weights: inverse class frequency per fold (A-E)
    ug, uc             = np.unique(y_train, return_counts=True)
    freq_map           = dict(zip(ug, uc))
    n_kelas            = len(ug)
    raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in y_train])
    sample_w           = raw_w / raw_w.mean()
    print(f"  Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}")

    # ── Ensemble: N_SEEDS run per fold ────────────────────────────────────────
    all_proba = []

    for seed in range(N_SEEDS):
        print(f"\n  -- Seed {seed+1}/{N_SEEDS} --")
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model = build_model_clf(
            q_seq_len    = questions_emb.shape[1],
            ak_seq_len   = answerkeys_emb.shape[1],
            a_seq_len    = answers_emb.shape[1],
            emb_dim      = answers_emb.shape[2],
            bilstm_units = BILSTM_UNITS,
            dropout      = DROPOUT,
            dense1       = DENSE1,
            dense2       = DENSE2
        )

        reduce_lr  = ReduceLROnPlateau(
            monitor='val_accuracy', factor=0.5, patience=3,
            min_lr=1e-6, verbose=0, mode='max'
        )
        early_stop = EarlyStopping(
            monitor='val_accuracy', patience=PATIENCE,
            restore_best_weights=True, verbose=1, mode='max'
        )

        model.fit(
            [X_q_tr, X_ak_tr, X_a_tr], y_train,
            sample_weight=sample_w,
            validation_data=([X_q_val, X_ak_val, X_a_val], y_val),
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            callbacks=[reduce_lr, early_stop], verbose=1
        )

        proba = model.predict([X_q_te, X_ak_te, X_a_te], verbose=0)  # (n_test, 5)
        all_proba.append(proba)

        pred_s = np.argmax(proba, axis=1)
        print(f"  Seed {seed+1} Accuracy: {accuracy_score(y_test, pred_s):.4f}  "
              f"F1(w): {f1_score(y_test, pred_s, average='weighted', zero_division=0):.4f}")

    # ── Rata-rata ensemble probabilitas -> argmax ──────────────────────────────
    avg_proba  = np.mean(all_proba, axis=0)        # (n_test, 5)
    y_pred     = np.argmax(avg_proba, axis=1)      # 0-4 (E..A)

    acc    = accuracy_score(y_test, y_pred)
    f1_w   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_mac = f1_score(y_test, y_pred, average='macro',    zero_division=0)
    # Jarak ordinal antar kelas sebagai pengganti MAE
    ord_err    = np.mean(np.abs(y_test.astype(int) - y_pred.astype(int)))
    adj_acc    = np.mean(np.abs(y_test.astype(int) - y_pred.astype(int)) <= 1)

    print(f"\n  Ensemble Acc: {acc:.4f}  F1(w): {f1_w:.4f}  F1(macro): {f1_mac:.4f}")
    print(f"  Ordinal Err: {ord_err:.4f}  Adj Acc (+-1 kelas): {adj_acc:.4f}")

    fold_results.append({
        'fold'       : i + 1,
        'test_idpsj' : test_id,
        'val_idpsj'  : val_id,
        'n_train'    : len(y_train),
        'n_val'      : len(y_val),
        'n_test'     : len(y_test),
        'accuracy'   : acc,
        'f1_weighted': f1_w,
        'f1_macro'   : f1_mac,
        'ord_err'    : ord_err,
        'adj_acc'    : adj_acc,
        'y_test'     : y_test,
        'y_pred'     : y_pred,
    })

    model_path = os.path.join(OUT_DIR, f'model_fold_{i+1:02d}_ae.keras')
    model.save(model_path)
    print(f"  Model (seed {N_SEEDS}) saved -> {model_path}")

    tf.keras.backend.clear_session()

print("\n\nSelesai semua fold.")

In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
summary = pd.DataFrame([{
    'fold'        : r['fold'],
    'test_idpsj'  : r['test_idpsj'],
    'n_train'     : r['n_train'],
    'n_test'      : r['n_test'],
    'Accuracy'    : round(r['accuracy'],    4),
    'F1_weighted' : round(r['f1_weighted'], 4),
    'F1_macro'    : round(r['f1_macro'],    4),
    'Ord_Err'     : round(r['ord_err'],     4),
    'Adj_Acc+-1'  : round(r['adj_acc'],     4),
} for r in fold_results])

print("=" * 75)
print("Hasil per Fold (Klasifikasi A-E)")
print("=" * 75)
print(summary.to_string(index=False))
print(f"\nRata-rata  Accuracy   : {summary['Accuracy'].mean():.4f} +- {summary['Accuracy'].std():.4f}")
print(f"Rata-rata  F1 (w)     : {summary['F1_weighted'].mean():.4f} +- {summary['F1_weighted'].std():.4f}")
print(f"Rata-rata  F1 (macro) : {summary['F1_macro'].mean():.4f} +- {summary['F1_macro'].std():.4f}")
print(f"Rata-rata  Ord Err    : {summary['Ord_Err'].mean():.4f} +- {summary['Ord_Err'].std():.4f}")
print(f"Rata-rata  Adj Acc+-1 : {summary['Adj_Acc+-1'].mean():.4f} +- {summary['Adj_Acc+-1'].std():.4f}")

summary.to_csv(os.path.join(OUT_DIR, 'lopo_results_ae.csv'), index=False)

# ── Classification Report gabungan semua fold ─────────────────────────────────
y_all_true = np.concatenate([r['y_test'] for r in fold_results])
y_all_pred = np.concatenate([r['y_pred'] for r in fold_results])

class_names = [IDX_TO_LABEL[i] for i in range(N_CLASSES)]
print("\nClassification Report (gabungan semua fold):")
print(classification_report(y_all_true, y_all_pred,
                             target_names=class_names, zero_division=0))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm  = confusion_matrix(y_all_true, y_all_pred, labels=list(range(N_CLASSES)))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(class_names, fontsize=12)
ax.set_yticklabels(class_names, fontsize=12)
ax.set_xlabel('Label Prediksi', fontsize=12)
ax.set_ylabel('Label Aktual', fontsize=12)
ax.set_title('Confusion Matrix — Siamese BiLSTM Klasifikasi A-E (semua fold)')
plt.colorbar(im)
for row in range(N_CLASSES):
    for col in range(N_CLASSES):
        ax.text(col, row, cm[row, col], ha='center', va='center', fontsize=11,
                color='white' if cm[row, col] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_matrix_ae.png'), dpi=150)
plt.show()

# ── Bar plot metrik per fold ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, metric in zip(axes, ['Accuracy', 'F1_weighted', 'Adj_Acc+-1']):
    ax.bar(summary['test_idpsj'].astype(str), summary[metric],
           color='steelblue', edgecolor='black')
    ax.axhline(summary[metric].mean(), color='red', linestyle='--',
               label=f'Mean {metric}')
    ax.set_title(f'{metric} per Fold (test IDPSJ)')
    ax.set_xlabel('Test IDPSJ')
    ax.set_ylabel(metric)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'lopo_metrics_ae.png'), dpi=150)
plt.show()